# 01 Data Cleaning & Analyst-Ready Dataset


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data/raw')

customers = pd.read_csv(DATA_DIR / 'customers.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
payments = pd.read_csv(DATA_DIR / 'payments.csv')
returns = pd.read_csv(DATA_DIR / 'returns.csv')
shipping = pd.read_csv(DATA_DIR / 'shipping.csv')

for name, df in {
    'customers': customers,
    'orders': orders,
    'products': products,
    'payments': payments,
    'returns': returns,
    'shipping': shipping,
}.items():
    print(f'\n{name.upper()}')
    print(df.head())
    print(df.info())


In [ ]:
# Convert dates
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
orders['order_date'] = pd.to_datetime(orders['order_date'])
returns['return_date'] = pd.to_datetime(returns['return_date'])
shipping['ship_date'] = pd.to_datetime(shipping['ship_date'])
shipping['delivery_date'] = pd.to_datetime(shipping['delivery_date'])

# Basic validation
print('Nulls by table:')
for name, df in {
    'customers': customers,
    'orders': orders,
    'products': products,
    'payments': payments,
    'returns': returns,
    'shipping': shipping,
}.items():
    print(name, df.isnull().sum().to_dict())


In [ ]:
# Join into one analyst-ready table
df = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(payments, on='order_id', how='left')
    .merge(shipping, on='order_id', how='left')
    .merge(returns[['order_id', 'reason']], on='order_id', how='left')
)

# Feature engineering
df['revenue'] = df['quantity'] * df['price'] - df['discount']
df['cost'] = df['quantity'] * df['cost_price']
df['gross_margin'] = df['revenue'] - df['cost']
df['return_flag'] = df['reason'].notna().astype(int)
df['delay_flag'] = (df['delivery_performance'] == 'Delayed').astype(int)
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)

df.head()


In [ ]:
# Cleaned export for downstream analysis
CLEAN_DIR = Path('../data/cleaned')
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_DIR / 'ecommerce_cleaned.csv', index=False)
print('Saved:', CLEAN_DIR / 'ecommerce_cleaned.csv')
